# Entrenamiento de una Red Profunda para CIFAR-10

Este notebook implementa un modelo profundo en Keras/TensorFlow para clasificación multiclase sobre CIFAR-10.

El conjunto CIFAR-10 contiene 60.000 imágenes pequeñas a color de 32x32 píxeles distribuidas en 10 clases: 50.000 para entrenamiento y 10.000 para prueba.

El modelo solicitado es una red densa profunda con 20 capas ocultas de 100 neuronas cada una, inicialización de He, activación Swish, Batch Normalization, Early Stopping, AdamW y un planificador de tasa de aprendizaje basado en desempeño.

## Objetivo

Construir, entrenar y evaluar una red profunda para CIFAR-10 mostrando `accuracy` y `AUC` tanto durante el entrenamiento como en la evaluación final.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

tf.keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)

print(tf.__version__)

I0000 00:00:1779978799.136085    6424 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1779978800.655512    6424 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779978805.845439    6424 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


2.21.0


## Carga y preparación de datos

Se usa `tf.keras.datasets.cifar10`. Las imágenes se normalizan al rango `[0, 1]` y las etiquetas se convierten a representación one-hot para trabajar con una salida `softmax`.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

x_train.shape, y_train.shape, x_test.shape, y_test.shape

## Arquitectura de la red

La red usa una estrategia MLP sobre imágenes aplanadas. Cada bloque oculto contiene una capa densa de 100 neuronas, seguida por Batch Normalization y activación Swish.

La inicialización He se aplica en las capas densas para favorecer una propagación estable del gradiente.

In [ ]:
def build_model(num_hidden_layers=20, units=100):
    he_init = tf.keras.initializers.HeNormal()
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Flatten()
    ])

    for _ in range(num_hidden_layers):
        model.add(tf.keras.layers.Dense(units, kernel_initializer=he_init, use_bias=False))
        model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.Activation("swish"))

    model.add(tf.keras.layers.Dense(10, activation="softmax", kernel_initializer=he_init))
    return model

model = build_model()
model.summary()

## Compilación

Se utiliza `AdamW` como optimizador. Para la métrica AUC se usa una variante multiclase compatible con las probabilidades generadas por la capa `softmax`.

In [ ]:
optimizer = tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4)
auc_metric = tf.keras.metrics.AUC(name="auc", multi_label=True, num_labels=10)

model.compile(
    optimizer=optimizer,
    loss="categorical_crossentropy",
    metrics=["accuracy", auc_metric],
)

model.metrics_names

## Regularización del entrenamiento

Se aplica Early Stopping para evitar sobreajuste y `ReduceLROnPlateau` como estrategia de performance scheduling.

In [ ]:
early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=8,
    restore_best_weights=True,
    verbose=1,
)

reduce_lr_cb = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_auc",
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1,
)

class LrHistory(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        logs["lr"] = float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))

lr_history_cb = LrHistory()

## Entrenamiento

Se reserva una fracción del conjunto de entrenamiento para validación interna. El modelo se entrena con batch normalization, Swish y callbacks para estabilizar la optimización.

In [ ]:
history = model.fit(
    x_train,
    y_train,
    validation_split=0.1,
    epochs=60,
    batch_size=128,
    callbacks=[early_stopping_cb, reduce_lr_cb, lr_history_cb],
    verbose=2,
)

## Curvas de entrenamiento

Se muestran las curvas principales para revisar el comportamiento del entrenamiento y la evolución de la tasa de aprendizaje.

In [ ]:
history_dict = history.history
epochs_ran = range(1, len(history_dict["loss"]) + 1)

plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.plot(epochs_ran, history_dict["loss"], label="train loss")
plt.plot(epochs_ran, history_dict["val_loss"], label="val loss")
plt.xlabel("epoch")
plt.title("Loss")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(epochs_ran, history_dict["accuracy"], label="train acc")
plt.plot(epochs_ran, history_dict["val_accuracy"], label="val acc")
plt.xlabel("epoch")
plt.title("Accuracy")
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(epochs_ran, history_dict["auc"], label="train auc")
plt.plot(epochs_ran, history_dict["val_auc"], label="val auc")
plt.plot(epochs_ran, history_dict["lr"], label="learning rate")
plt.xlabel("epoch")
plt.title("AUC / LR")
plt.legend()

plt.tight_layout()
plt.show()

## Evaluación final

Se evalúa el modelo sobre el conjunto de prueba y se reportan `accuracy` y `AUC`.

In [ ]:
test_loss, test_accuracy, test_auc = model.evaluate(x_test, y_test, verbose=0)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test AUC: {test_auc:.4f}")